# Python, NumPy, and pandas Refresher for ML

> Review the small set of coding patterns that appear throughout machine learning.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Use arrays and DataFrames for feature matrices and targets.
- Understand shape, indexing, boolean masks, and reproducible randomness.
- Avoid accidental mutation and row/target misalignment.

Most estimators expect `X` to contain features and `y` to contain the target. Shape mistakes and misaligned rows are common sources of silent bugs.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
X = rng.normal(size=(5, 3))  # 5 observations, 3 features
y = np.array([0, 1, 0, 1, 1])
print("X shape:", X.shape, "y shape:", y.shape)
print("First observation:", X[0])
print("Second feature for every row:", X[:, 1])
print("Positive rows:", X[y == 1])

X shape: (5, 3) y shape: (5,)
First observation: [ 0.30471708 -1.03998411  0.7504512 ]
Second feature for every row: [-1.03998411 -1.95103519 -0.31624259  0.87939797  1.12724121]
Positive rows: [[ 0.94056472 -1.95103519 -1.30217951]
 [-0.85304393  0.87939797  0.77779194]
 [ 0.0660307   1.12724121  0.46750934]]


## DataFrames and explicit selection
Use `.loc` for label-based selection and `.iloc` for position-based selection. Be careful with chained assignment; use a single `.loc` operation to change values.

In [2]:
df = pd.DataFrame({"age": [24, 31, 45, 38], "income": [28000, 41000, 72000, 51000], "city": ["A", "B", "A", "C"]})
df["income_k"] = df["income"] / 1000
print(df.loc[df["age"] >= 35, ["age", "income_k"]])
print("Numeric summary:")
print(df.select_dtypes("number").describe().round(1))

   age  income_k
2   45      72.0
3   38      51.0
Numeric summary:
        age   income  income_k
count   4.0      4.0       4.0
mean   34.5  48000.0      48.0
std     9.0  18565.2      18.6
min    24.0  28000.0      28.0
25%    29.2  37750.0      37.8
50%    34.5  46000.0      46.0
75%    39.8  56250.0      56.2
max    45.0  72000.0      72.0


## Vectorization versus loops
NumPy and pandas are optimized for column-wise operations. Prefer vectorized expressions for simple transformations. Loops remain useful when the logic is inherently sequential or calls an external service.

In [3]:
values = np.array([10., 20., 30., 40.])
print("Standardized (manual):", ((values - values.mean()) / values.std()).round(2))
print("Above mean:", values[values > values.mean()])

# A reproducible split of indices; do not sort X without applying the same order to y.
idx = rng.permutation(len(values))
print("Permutation:", idx, "aligned values:", values[idx])

Standardized (manual): [-1.34 -0.45  0.45  1.34]
Above mean: [30. 40.]
Permutation: [1 0 3 2] aligned values: [20. 10. 40. 30.]


## Common gotchas
- A one-dimensional target has shape `(n_samples,)`; feature matrices usually have shape `(n_samples, n_features)`.
- `df["target"]` returns a Series, while `df[["target"]]` returns a one-column DataFrame.
- `inplace=True` is not a universal speed improvement; explicit assignment is often clearer.
- Never shuffle features and labels independently.

In [4]:
print(type(df["age"]).__name__, type(df[["age"]]).__name__)
print("Rows:", len(df), "Unique cities:", df["city"].nunique())
assert len(X) == len(y), "Feature rows and targets must remain aligned."

Series DataFrame
Rows: 4 Unique cities: 3


## Practice
1. Create a mask for rows with income above 40,000.
2. Convert a numeric column to a NumPy array.
3. Explain what `X.shape[1]` represents.

In [5]:
print(df.loc[df["income"] > 40000, ["city", "income"]])
income_array = df["income"].to_numpy()
print(income_array, "number of features:", X.shape[1])

  city  income
1    B   41000
2    A   72000
3    C   51000
[28000 41000 72000 51000] number of features: 3
